# Similaridad (Videos)

## Librerias

In [ ]:
# ========= Data Cleaning =========

!pip install advertools
!pip install clean-text
!pip install urlextract
!pip install Counter
import advertools as adv
from urlextract import URLExtract
from nltk.tokenize import regexp_tokenize
import html
import string 
import re
from cleantext import clean
import unicodedata

import nltk
from nltk import word_tokenize
from nltk.probability import FreqDist
from matplotlib import pyplot as plt
import seaborn as sns
from wordcloud import WordCloud
from collections import Counter
from nltk.corpus import stopwords
nltk.download('stopwords')
nltk.download('punkt')

from sklearn.feature_extraction.text import CountVectorizer
import numpy as np
from numpy import dot
from numpy.linalg import norm

# ========= Data Cleaning =========

import pandas as pd
import requests
import csv
import time
from datetime import datetime
from datetime import date
import os.path
from os import path
from google.colab import drive
key='TU_CLAVE_DE_API'
drive.mount('/content/drive/')

## Importando datos

In [ ]:
# ================== OJO modificar para cada serie ==================

path='/content/drive/MyDrive/AI-GUMELAB/CognASSISTANT-1/SEARCHES/YoutubeResults/Videos/\
Narcos/2022-08-26/RESULTS-VIDEOS2022-08-26 23:39:06.922705.csv' 

# ================== OJO modificar para cada serie ==================

Videos = pd.read_csv(path)
Videos

In [ ]:
Videos.info()

In [ ]:
# Dataframe visual para exposiciones
Videos[['TituloVideo','TituloVideoModificado','emojisTituloVideo','URLsTituloVideo','hashtagsTituloVideo','MencionesTituloVideo','DescripcionVideo','DescripcionVideoModificado','emojisDescripcionVideo'\
        ,'URLsDescripcionVideo','hashtagsDescripcionVideo','MencionesDescripcionVideo','FechaPublicacionVideo','VistasVideo','LikesVideo','ComentariosVideo'\
        ,'EtiquetasVideo','EtiquetasVideoTexto','TituloCanalVideo','ConsultaVideo','ConsultaVideoModificado','PaisVideo','TotalTextoVideoModificado']].head(5)

In [ ]:
Videos['TituloVideoModificado']=Videos['TituloVideoModificado'].fillna('No Titulo')

In [ ]:
Videos.iloc[30:50]

In [ ]:
!pip install translators --upgrade

In [ ]:
import translators as ts

In [ ]:
frase=Videos['TituloVideoModificado'][30]

In [ ]:
frase

In [ ]:
ts.google(frase, to_language='es')

In [ ]:
def traduccion(texto):
  texto=ts.google(texto, to_language='es')
  return texto

In [ ]:
traduccion(frase)

In [ ]:
Videos['TotalTextoVideoModificado']=Videos['TotalTextoVideoModificado'].apply(traduccion)

## Prueba de Vocabulario

### Oracion completa y tokens

In [ ]:
sentenceTituloVideoModificado =" ".join(Videos['TituloVideoModificado'])
sentenceTituloVideoModificado 

In [ ]:
tokensTituloVideoModificado = word_tokenize(sentenceTituloVideoModificado)
len(tokensTituloVideoModificado)

### Oracion completa en Consulta y tokens

In [ ]:
sentenceConsultaVideoModificado =" ".join(Videos['ConsultaVideoModificado'])
sentenceConsultaVideoModificado

In [ ]:
tokensConsultaVideoModificado = word_tokenize(sentenceConsultaVideoModificado)
len(tokensConsultaVideoModificado)

## Conteo de palabras

In [ ]:
counted = Counter(tokensTituloVideoModificado)
word_freqTotalTextDesc = pd.DataFrame(counted.items(),columns=['Palabra','Frecuencia'] ).sort_values(by='Frecuencia',ascending=False, ignore_index=True)
word_freqTotalTextAsc = pd.DataFrame(counted.items(),columns=['Palabra','Frecuencia'] ).sort_values(by='Frecuencia',ascending=True, ignore_index=True)

In [ ]:
word_freqTotalTextDesc

In [ ]:
word_freqTotalTextAsc

## Importando datos (Solo si no es la primera vez)

In [ ]:
# ================== OJO modificar para cada serie ==================

path='/content/drive/MyDrive/AI-GUMELAB/CognASSISTANT-1/SEARCHES/YoutubeResults/Similaridad/\
Narcos/2022-08-27/VIDEOS+Cos(TF_IDF)2Titulo2022-08-27 01:06:19.300259.csv' 

# ================== OJO modificar para cada serie ==================

Videos = pd.read_csv(path)
Videos

## Representacion de texto o vectorizacion

In [ ]:
def cossmimilaridad(T,C):
  similaridad=[]
  for i in range(len(T)):
    similaridad.append(dot(T[:][i], C[:][i])/(norm(T[:][i])*norm(C[:][i])))
  return similaridad

### Bolsa de palabras

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
vect = CountVectorizer()

In [ ]:
vect.fit(Videos['TituloEtiquetasModificado'])

In [ ]:
print("Vocabulary size: {}".format(len(vect.vocabulary_)))
print("Vocabulary content:\n {}".format(vect.vocabulary_))

In [ ]:
bag_of_wordsTextoTotal = vect.transform(Videos['TituloEtiquetasModificado'])
print("bag_of_words: {}".format(repr(bag_of_wordsTextoTotal)))

In [ ]:
bag_of_wordsConsulta = vect.transform(Videos['ConsultaVideoModificado'])
print("bag_of_words: {}".format(repr(bag_of_wordsConsulta)))

In [ ]:
T=bag_of_wordsTextoTotal.toarray()
C=bag_of_wordsConsulta.toarray()

In [ ]:
similaridad=cossmimilaridad(T,C)

In [ ]:
len(similaridad)

In [ ]:
SC=pd.Series(similaridad, name='Cos(BolsaPalabras)')
SC

In [ ]:
Videospuntaje=pd.concat([Videos,SC],axis=1)
Videospuntaje

#### Carpetas Drive

In [ ]:
today = str(date.today())

# ======================= OJO Cambiar PATH para cada Serie =======================

PATH_TO_SAVE_SEARCHES = '/content/drive/MyDrive/AI-GUMELAB/CognASSISTANT-1/SEARCHES/YoutubeResults/Similaridad/Narcos/'

# ======================= OJO Cambiar PATH para cada Serie =======================

PATH_TO_DIR_0 = PATH_TO_SAVE_SEARCHES + today
if os.path.exists(PATH_TO_DIR_0) == False:
  os.mkdir(PATH_TO_DIR_0)
file_nameL = 'VIDEOS+Cos(BolsaPalabras)1TituloEtiquetas'+str(datetime.now())

In [ ]:
Videospuntaje.to_csv(PATH_TO_DIR_0+'/'+file_nameL+'.csv', index=False)

### Representacion TF-IDF

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf = TfidfVectorizer()

In [ ]:
tfidf.fit(Videos['TituloVideoModificado'])

In [ ]:
print("Vocabulary size: {}".format(len(tfidf.vocabulary_)))
print("Vocabulary content:\n {}".format(tfidf.vocabulary_))

In [ ]:
TF_IDFTextoTotal = tfidf.transform(Videos['TituloVideoModificado'])
print("bag_of_words: {}".format(repr(TF_IDFTextoTotal)))

In [ ]:
TF_IDFConsulta = tfidf.transform(Videos['ConsultaVideoModificado'])
print("bag_of_words: {}".format(repr(TF_IDFConsulta)))

In [ ]:
T=TF_IDFTextoTotal.toarray()
C=TF_IDFConsulta.toarray()

In [ ]:
similaridad=cossmimilaridad(T,C)

In [ ]:
len(similaridad)

In [ ]:
SC=pd.Series(similaridad, name='Cos(TF_IDF)')
SC

In [ ]:
Videospuntaje=pd.concat([Videos,SC],axis=1)
Videospuntaje

#### Carpetas Drive

In [ ]:
today = str(date.today())

# ======================= OJO Cambiar PATH para cada Serie =======================

PATH_TO_SAVE_SEARCHES = '/content/drive/MyDrive/AI-GUMELAB/CognASSISTANT-1/SEARCHES/YoutubeResults/Similaridad/Narcos/'

# ======================= OJO Cambiar PATH para cada Serie =======================

PATH_TO_DIR_0 = PATH_TO_SAVE_SEARCHES + today
if os.path.exists(PATH_TO_DIR_0) == False:
  os.mkdir(PATH_TO_DIR_0)
file_nameL = 'VIDEOS+Cos(TF_IDF)2Titulo'+str(datetime.now())

In [ ]:
Videospuntaje.to_csv(PATH_TO_DIR_0+'/'+file_nameL+'.csv', index=False)

### n- grams

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
count_vect = CountVectorizer(ngram_range=(1,3))

In [ ]:
count_vect.fit(Videos['TituloVideoModificado'])

In [ ]:
print("Vocabulary size: {}".format(len(count_vect.vocabulary_)))
print("Vocabulary content:\n {}".format(count_vect.vocabulary_))

In [ ]:
NgramsTextoTotal = count_vect.transform(Videos['TituloVideoModificado'])
print("bag_of_words: {}".format(repr(NgramsTextoTotal)))

In [ ]:
NgramsConsulta = count_vect.transform(Videos['ConsultaVideoModificado'])
print("bag_of_words: {}".format(repr(NgramsConsulta)))

In [ ]:
T=NgramsTextoTotal.toarray()
C=NgramsConsulta.toarray()

In [ ]:
similaridad=cossmimilaridad(T,C)

In [ ]:
len(similaridad)

In [ ]:
SC=pd.Series(similaridad, name='Cos(Ngrams)')
SC

In [ ]:
Videospuntaje=pd.concat([Videospuntaje,SC],axis=1)
Videospuntaje

#### Carpetas Drive

In [ ]:
today = str(date.today())

# ======================= OJO Cambiar PATH para cada Serie =======================

PATH_TO_SAVE_SEARCHES = '/content/drive/MyDrive/AI-GUMELAB/CognASSISTANT-1/SEARCHES/YoutubeResults/Similaridad/Narcos/'

# ======================= OJO Cambiar PATH para cada Serie =======================

PATH_TO_DIR_0 = PATH_TO_SAVE_SEARCHES + today
if os.path.exists(PATH_TO_DIR_0) == False:
  os.mkdir(PATH_TO_DIR_0)
file_nameL = 'VIDEOS+Cos(TF_IDF)3Titulo'+str(datetime.now())

In [ ]:
Videospuntaje.to_csv(PATH_TO_DIR_0+'/'+file_nameL+'.csv', index=False)

### Analisando nuevas columnas

In [ ]:
Videospuntaje[['Cos(BolsaPalabras)','Cos(TF_IDF)','Cos(Ngrams)']].hist(figsize=(10,10));

In [ ]:
Videospuntaje

In [ ]:
plt.figure(figsize=(10,12))
sns.boxplot(data=Videospuntaje[['Cos(BolsaPalabras)','Cos(TF_IDF)','Cos(Ngrams)']])
sns.stripplot(data=Videospuntaje[['Cos(BolsaPalabras)','Cos(TF_IDF)','Cos(Ngrams)']], color="pink", jitter=True, size=8, alpha=0.6)
plt.show()

In [ ]:
Videospuntaje

## Ordenando datos

In [ ]:
Videospuntaje.info()

In [ ]:
Videospuntaje.sort_values(by=['Cos(TF_IDF)','ComentariosVideo'], inplace=True, ascending=False)
Videospuntaje

## Filtrando

In [ ]:
filtro = Videospuntaje[Videospuntaje['Cos(TF_IDF)'] <= float(0.000)].assign(Relevancia='No relevante')

In [ ]:
filtro

In [ ]:
filtro = Videospuntaje[Videospuntaje['Cos(BolsaPalabras)'] <= float(0.17000)].assign(Relevancia='No relevante')
filtro['Cos(BolsaPalabras)']=filtro['Cos(BolsaPalabras)'].astype(str)
filtro['Cos(BolsaPalabras)'] =filtro['Cos(BolsaPalabras)'].str.replace('.', ',',regex=True)
filtro['Cos(TF_IDF)']=filtro['Cos(TF_IDF)'].astype(str)
filtro['Cos(TF_IDF)'] =filtro['Cos(TF_IDF)'].str.replace('.', ',',regex=True)
filtro['Cos(Ngrams)']=filtro['Cos(Ngrams)'].astype(str)
filtro['Cos(Ngrams)'] =filtro['Cos(Ngrams)'].str.replace('.', ',',regex=True)

In [ ]:
filtro

In [ ]:
filtro_ids=filtro[['idRecursoVD','Relevancia']]
filtro_ids

In [ ]:
Videospuntaje['Cos(BolsaPalabras)']=Videospuntaje['Cos(BolsaPalabras)'].astype(str)
Videospuntaje['Cos(BolsaPalabras)'] =Videospuntaje['Cos(BolsaPalabras)'].str.replace('.', ',',regex=True)
Videospuntaje['Cos(TF_IDF)']=Videospuntaje['Cos(TF_IDF)'].astype(str)
Videospuntaje['Cos(TF_IDF)'] =Videospuntaje['Cos(TF_IDF)'].str.replace('.', ',',regex=True)
Videospuntaje['Cos(Ngrams)']=Videospuntaje['Cos(Ngrams)'].astype(str)
Videospuntaje['Cos(Ngrams)'] =Videospuntaje['Cos(Ngrams)'].str.replace('.', ',',regex=True)

In [ ]:
VideospuntajeN = pd.merge(left=Videospuntaje, right=filtro_ids, how="left", left_on='idRecursoVD', right_on='idRecursoVD')
VideospuntajeN

In [ ]:
VideospuntajeN = VideospuntajeN[VideospuntajeN['Relevancia'] != 'No relevante']
VideospuntajeN

In [ ]:
VideospuntajeN.info()

## Carpetas Drive

In [ ]:
today = str(date.today())

# ======================= OJO Cambiar PATH para cada Serie =======================

PATH_TO_SAVE_SEARCHES = '/content/drive/MyDrive/AI-GUMELAB/CognASSISTANT-1/SEARCHES/YoutubeResults/DataAnalisisVideos/Narcos/'

# ======================= OJO Cambiar PATH para cada Serie =======================

PATH_TO_DIR_0 = PATH_TO_SAVE_SEARCHES + today
if os.path.exists(PATH_TO_DIR_0) == False:
  os.mkdir(PATH_TO_DIR_0)
file_nameL = 'VIDEOS'+str(datetime.now())

In [ ]:
filtro.to_csv(PATH_TO_DIR_0+'/'+'filtro-'+file_nameL+'.csv', index=False, decimal=',')
VideospuntajeN.to_csv(PATH_TO_DIR_0+'/'+'RESULTS-'+file_nameL+'.csv', index=False)